# Implementación de Regresión Lineal y Logística usando Scikit-Learn
## SI3015 - Fundamentos de Aprendizaje Automático

Este cuaderno presenta la resolución de los ejercicios propuestos para la Clase #5, utilizando el dataset del Titanic.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge, Lasso, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score
from scipy.stats import uniform

# Configuración de visualización
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

### 1. Carga y Exploración de Datos

In [ ]:
# Cargar el conjunto de datos
df = pd.read_csv('Titanic-Dataset.csv')

# Visualizar las primeras filas
display(df.head())

# Información general
df.info()

#### Limpieza de Datos
Basado en la exploración, manejaremos valores nulos y realizaremos transformaciones.

In [ ]:
# Imputar valores faltantes
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df.drop(columns=['Cabin', 'Name', 'Ticket', 'PassengerId'], inplace=True)

# Exploración gráfica
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(df['Age'], kde=True, ax=axes[0]).set_title('Distribución de Edad')
sns.countplot(data=df, x='Survived', ax=axes[1]).set_title('Conteo de Supervivientes')
plt.show()

### 1.0.1. Regresión Lineal
Determinamos que la columna objetivo será `Age` (aunque el objetivo principal del dataset es supervivencia, usaremos `Age` para el ejercicio de regresión lineal basado en otras características como `Pclass`, `Fare`, `SibSp`, `Parch`).

In [ ]:
# Definir características y objetivo
X_lin = df.drop(columns=['Age'])
y_lin = df['Age']

# Dividir el dataset
X_train_lin, X_test_lin, y_train_lin, y_test_lin = train_test_split(X_lin, y_lin, test_size=0.2, random_state=42)

print(f"Entrenamiento: {len(X_train_lin)} muestras")
print(f"Prueba: {len(X_test_lin)} muestras")

#### Definición de Pipelines y Búsqueda
Definimos el preprocesamiento para variables categóricas (`Sex`, `Embarked`) y numéricas.

In [ ]:
numeric_features = ['Fare', 'SibSp', 'Parch', 'Pclass', 'Survived']
categorical_features = ['Sex', 'Embarked']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(), categorical_features)
    ])

# Pipelines
ridge_pipeline = Pipeline([('preprocessor', preprocessor), ('regressor', Ridge())])
lasso_pipeline = Pipeline([('preprocessor', preprocessor), ('regressor', Lasso())])

# Parámetros para búsqueda aleatoria
param_dist = {
    'regressor__alpha': uniform(0, 10)
}

# Búsqueda Aleatoria y Cross-Validation
ridge_search = RandomizedSearchCV(ridge_pipeline, param_dist, n_iter=50, cv=5, scoring='neg_mean_absolute_error', random_state=42)
lasso_search = RandomizedSearchCV(lasso_pipeline, param_dist, n_iter=50, cv=5, scoring='neg_mean_absolute_error', random_state=42)

ridge_search.fit(X_train_lin, y_train_lin)
lasso_search.fit(X_train_lin, y_train_lin)

print("Mejor alpha Ridge:", ridge_search.best_params_)
print("Mejor alpha Lasso:", lasso_search.best_params_)

#### Evaluación de Modelos Lineales

In [ ]:
def evaluate_lin(model, X_test, y_test, name):
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    print(f"--- {name} ---")
    print(f"R^2: {r2:.4f}")
    print(f"MAE: {mae:.4f}")
    return y_pred

y_pred_ridge = evaluate_lin(ridge_search, X_test_lin, y_test_lin, "Ridge")
y_pred_lasso = evaluate_lin(lasso_search, X_test_lin, y_test_lin, "Lasso")

#### Gráficos de Resultados de Regresión Lineal

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(y_test_lin, y_pred_ridge, alpha=0.5, label='Ridge Pred vs Actual', color='blue')
plt.scatter(y_test_lin, y_pred_lasso, alpha=0.3, label='Lasso Pred vs Actual', color='red')
plt.plot([y_test_lin.min(), y_test_lin.max()], [y_test_lin.min(), y_test_lin.max()], 'k--', lw=2)
plt.xlabel('Real')
plt.ylabel('Predicho')
plt.legend()
plt.title('Regresión Lineal: Reales vs Predichos')
plt.show()

### 1.0.2. Regresión Logística
Columnas objetivo: `Survived`.

In [ ]:
X_log = df.drop(columns=['Survived'])
y_log = df['Survived']

X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(X_log, y_log, test_size=0.2, random_state=42)

# Reutilizamos el preprocesador pero ajustado para X_log
num_features_log = ['Age', 'Fare', 'SibSp', 'Parch', 'Pclass']
cat_features_log = ['Sex', 'Embarked']
preprocessor_log = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features_log),
        ('cat', OneHotEncoder(), cat_features_log)
    ])

log_pipeline = Pipeline([('preprocessor', preprocessor_log), ('classifier', LogisticRegression())])

param_dist_log = {
    'classifier__C': uniform(0.1, 10),
    'classifier__penalty': ['l2'] 
}

log_search = RandomizedSearchCV(log_pipeline, param_dist_log, n_iter=30, cv=5, scoring='accuracy', random_state=42)
log_search.fit(X_train_log, y_train_log)

print("Mejores parámetros Logística:", log_search.best_params_)

#### Evaluación de Regresión Logística

In [ ]:
y_pred_log = log_search.predict(X_test_log)
acc = accuracy_score(y_test_log, y_pred_log)
f1 = f1_score(y_test_log, y_pred_log)

print(f"Accuracy: {acc:.4f}")
print(f"F1-score: {f1:.4f}")

# Matriz de Confusión
cm = confusion_matrix(y_test_log, y_pred_log)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Sobrevive', 'Sobrevive'])
disp.plot(cmap='Blues')
plt.title('Matriz de Confusión - Regresión Logística')
plt.show()

### 1.1. Funcionalidades Extra
#### 1.1.1. Importancia de Características (Feature Importance)
Analizaremos qué variables influyen más en la predicción de supervivencia.

In [ ]:
def plot_feature_importance(model, preprocessor, features_num, features_cat, title):
    ohe = preprocessor.named_transformers_['cat']
    cat_names = ohe.get_feature_names_out(features_cat)
    all_features = np.concatenate([features_num, cat_names])
    
    coefs = model.best_estimator_.named_steps['classifier'].coef_[0]
    
    importance_df = pd.DataFrame({'Feature': all_features, 'Coefficient': coefs})
    importance_df = importance_df.reindex(importance_df.Coefficient.abs().sort_values(ascending=False).index)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=importance_df, x='Coefficient', y='Feature', palette='viridis')
    plt.title(title)
    plt.axvline(0, color='black', linestyle='--')
    plt.show()

plot_feature_importance(log_search, preprocessor_log, num_features_log, cat_features_log, "Importancia de Características - Regresión Logística")

#### 1.1.2. Curva ROC (Receiver Operating Characteristic)
Evaluamos la capacidad de discriminación del modelo logístico a diferentes umbrales.

In [ ]:
y_probs = log_search.predict_proba(X_test_log)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test_log, y_probs)
auc = roc_auc_score(y_test_log, y_probs)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc:.4f})', color='darkorange', lw=2)
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
plt.title('Curva ROC - Supervivencia Titanic')
plt.legend(loc="lower right")
plt.show()